In [ ]:
#| include: false
![](https://capsule-render.vercel.app/api?type=waving&color=008080&height=300&section=header&text=fasterai%20&fontSize=90&animation=fadeIn&fontAlignY=38&desc=A%20Library%20to%20make%20smaller%20and%20faster%20neural%20networks&descAlignY=51&descAlign=62)


<p align="center">
    <a href="https://pypi.org/project/fasterai/"><img src="https://img.shields.io/pypi/v/fasterai?color=00a89e"></a>
    <a href="https://www.apache.org/licenses/LICENSE-2.0"><img src="https://img.shields.io/github/license/nathanhubens/fasterai?color=00a89e"></a>
    <a href="https://pypi.org/project/fasterai/"><img src="https://img.shields.io/badge/DOI-10.5281%2Fzenodo.6469868-y?color=00a89e"></a>
</p>


<p align="center">
  <a href="#features">Features</a> •
  <a href="#installation">Installation</a> •
  <a href="#tutorials">Tutorials</a> •
  <a href="#citing">Citing</a> •
  <a href="#license">License</a>
</p>

`fasterai` is a library created to make neural network **smaller** and **faster**. It gathers the
common compression techniques — sparsification, pruning, regularization, knowledge distillation,
quantization — and the steps that follow them: exporting the compressed model, and measuring which
layers can take the compression.

Each technique is built around the same four modules: **granularity**, **context**, **criteria**,
**schedule**. Each of them is customizable, so you can change them according to your needs or come up
with your own.

## Project Documentation

Visit the [Read The Docs Project Page](https://FasterAI-Labs.github.io/fasterai/) or read the
following README to know more about using `fasterai`.

---

##  Features

### 1. Sparsifying

![](imgs/sparsification.png)

Make your model sparse according to a: <br>
- <b>Sparsity: </b> the fraction of weights that will be replaced by 0 <br>
- <b>Granularity: </b> the granularity at which you operate the sparsification (weights, vectors, kernels, filters) <br>
- <b>Context: </b> sparsify either each layer independently (local) or the whole model (global) <br>
- <b>Criteria: </b> the criteria used to select the weights to remove (magnitude, movement, ...) <br>
- <b>Schedule: </b> which schedule you want to follow (one shot, iterative, gradual, ...) <br>

```python
SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                 criteria=large_final, schedule=one_cycle)
```

### 2. Pruning

![](imgs/pruning_readme.png "Pruning")

Once your model has useless nodes due to zero-weights, they can be removed to not be a part of the
network anymore. The model that comes out is structurally smaller.

```python
PruneCallback(pruning_ratio=0.3, schedule=one_cycle, context='global', criteria=large_final)
```

### 3. Regularization

![](imgs/regularization.png "Regularization")

Instead of explicitly making your network sparse, let it train towards sparse connections by
pushing the weights of a group to be as small as possible. Regularization follows the same
granularities as sparsifying.

```python
RegularizeCallback(criteria=large_final, granularity='filter', weight=0.01)
```

### 4. Knowledge Distillation

![alt text](imgs/distillation.png "Distillation")

Distill the knowledge acquired by a big model into a smaller one, comparing their predictions or
their intermediate activations.

```python
KnowledgeDistillationCallback(teacher, loss=SoftTarget, weight=0.5)
```

### 5. Lottery Ticket Hypothesis

![](imgs/LTH.png "Lottery Ticket Hypothesis")

Find the winning ticket in your network, *i.e.* re-train a sparse subnetwork from the initial weights
it started with.

```python
SparsifyCallback(sparsity=0.5, granularity='weight', context='local', criteria=large_final,
                 schedule=iterative, lth=True, rewind_epoch=1)
```

### 6. Quantization

Lower the arithmetic to INT8, either after training on calibration data or during it. The
precision is named argument by argument, and a backend that cannot honor it refuses instead of
quantizing something else.

```python
Quantizer(backend='pt2e', method='static', weight_bits=8, act_bits=8, qscheme='per_channel',
          symmetric=True).quantize(model, calibration_dl)
```

### 7. Export

Write the compressed model as an ONNX file, then read back what the exporter produced rather than
what it was asked for.

```python
path = export_qdq(qmodel, sample, 'model.onnx')
qdq_stats(path)                       # Q/DQ nodes, per-channel scales, zero-points
verify_qdq(qmodel, path, samples)     # argmax agreement with PyTorch
```

### 8. Analysis

Measure how much each layer can take before compressing the whole model.

```python
analyze_sensitivity(model, sample, eval_fn, compression='pruning', level=0.5)
```

### 9. Architecture rewrites

Fold batch norm into the preceding convolution, factorize fully-connected or convolution layers,
or prepare a model for CPU inference.

```python
BN_Folder().fold(model)
FC_Decomposer().decompose(model)
Conv_Decomposer().decompose(model, method='tucker')
optimize_for_cpu(model, sample)
```

---

## The FasterAI Ecosystem

fasterai is part of a family of libraries designed to make neural network optimization accessible:

| Package | Purpose | Scope |
|---------|---------|-------|
| **fasterai** | Compression techniques | Pruning, sparsification, distillation, quantization during training |
| **fasterbench** | Benchmarking | Measuring model size, speed, memory, compute, and energy |
| **fasterlatency** | Latency prediction | Hardware-aware neural architecture search |

### Typical Workflow

1. **Benchmark** your model with `fasterbench` to identify bottlenecks
2. **Compress** using `fasterai` techniques (pruning, distillation, quantization)
3. **Validate** compression impact with `fasterbench`
4. **Deploy** the optimized model

##  Quick Start

### 0. Import fasterai

```python
from fasterai.sparse.all import *
```

### 1. Create your model with fastai

```python
learn = vision_learner(dls, resnet18, metrics=accuracy)
```

### 2. Get your fasterai callback

```python
sp_cb = SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                         criteria=large_final, schedule=one_cycle)
```

### 3. Train your model to make it sparse !

```python
learn.fit_one_cycle(3, cbs=sp_cb)
```

Compression ratios are fractions in [0, 1]: `sparsity=0.5` zeroes half of the weights.

---

##  Installation

```sh
pip install git+https://github.com/FasterAI-Labs/fasterai.git
```

or 

```sh
pip install fasterai
```

---

## Tutorials

- [Get Started with FasterAI](tutorials/walkthrough.html)
- [Sparsify a model while it trains](tutorials/sparse/sparsify_callback.html)
- [Prune filters during training](tutorials/prune/prune_callback.html)
- [Export a deployable INT8 model](tutorials/quantize/deployable_export.html)
- [Create your own pruning schedule](tutorials/sparse/schedules.html)
- [Find winning tickets using the Lottery Ticket Hypothesis](tutorials/sparse/lottery_ticket.html)
- [Use Knowledge Distillation to help a student model to reach higher performance](tutorials/distill/distill_callback.html)
- [Sparsify Transformers](tutorials/sparse/transformers.html)

---

##  Citing
```latex
@software{Hubens,
  author       = {Nathan Hubens},
  title        = {fasterai},
  year         = 2022,
  publisher    = {Zenodo},
  version      = {v0.3.3},
  doi          = {10.5281/zenodo.6469868},
  url          = {https://doi.org/10.5281/zenodo.6469868}
}
```

---

## License

[Apache-2.0](https://www.apache.org/licenses/) License.

In [ ]:
#| include: false
![](https://capsule-render.vercel.app/api?type=waving&color=008080&height=100&section=footer)